In [1]:
import pandas as pd

Połączenie danych z MovieLens i z TMDB pobramych dla zbioru MovieLens

In [2]:
movie_lens = pd.read_csv('../data/merged/movielens.csv')
movie_lens

,movieId,imdbId,tmdbId,title,year,genres,top_5_tags
0,1,114709,862.0,Toy Story,1995.0,Adventure|Animation|Children|Comedy|Fantasy,"toys, computer animation, pixar animation, ani..."
1,2,113497,8844.0,Jumanji,1995.0,Adventure|Children|Fantasy,"adventure, children, fantasy, kids, childhood"
2,3,113228,15602.0,Grumpier Old Men,1995.0,Comedy|Romance,"good sequel, sequel, sequels, comedy, original"
3,4,114885,31357.0,Waiting to Exhale,1995.0,Comedy|Drama|Romance,"women, chick flick, girlie movie, romantic, di..."
4,5,113041,11862.0,Father of the Bride Part II,1995.0,Comedy,"good sequel, sequel, sequels, midlife crisis, ..."
...,...,...,...,...,...,...,...
86532,288967,14418234,845861.0,State of Siege: Temple Attack,2021.0,Action|Drama,NaN
86533,288971,11162178,878958.0,Ouija Japan,2021.0,Action|Horror,NaN
86534,288975,70199,150392.0,The Men Who Made the Movies: Howard Hawks,1973.0,Documentary,NaN
86535,288977,23050520,1102551.0,Skinford: Death Sentence,2023.0,Crime|Thriller,NaN


In [3]:
tmdb = pd.read_csv('../data/tmdb/tmdb_data_from_movieLens.csv')
tmdb

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,genres,origin_countries,spoken_languages
0,862.0,Toy Story,30000000,401157969,1995-11-22,81,en,21.5820,7.971,19651,"Family, Comedy, Animation, Adventure",United States of America,English
1,8844.0,Jumanji,65000000,262821940,1995-12-15,104,en,3.1836,7.245,11172,"Adventure, Fantasy, Family",United States of America,"English, Français"
2,15602.0,Grumpier Old Men,25000000,71518503,1995-12-22,101,en,2.1246,6.500,420,"Romance, Comedy",United States of America,English
3,31357.0,Waiting to Exhale,16000000,81452156,1995-12-22,127,en,2.0018,6.253,194,"Comedy, Drama, Romance",United States of America,English
4,11862.0,Father of the Bride Part II,0,76594107,1995-12-08,106,en,2.5346,6.275,795,"Comedy, Family",United States of America,English
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57703,385377.0,Ballet of Blood,0,0,2016-03-01,98,en,0.3408,1.800,10,Horror,United States of America,English
57704,644612.0,Letters Of Happiness,0,0,2019-12-03,103,ru,0.7063,6.300,9,"Drama, Family",Russia,Pусский
57705,845861.0,Akshardham: Operation Vajra Shakti,0,0,2025-07-04,106,hi,1.2825,10.000,2,"Action, Drama",India,हिन्दी
57706,878958.0,Ouija Japan,0,0,2021-10-19,78,en,0.6105,2.000,1,"Action, Horror",NaN,"日本語, English"


In [4]:
df = pd.merge(
    tmdb,
    movie_lens.drop(columns=['movieId', 'title']),
    how='left',
    on='tmdbId').rename(columns={'top_5_tags': 'keywords'})
df['release_date'] = pd.to_datetime(df['release_date'])
df

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,genres_x,origin_countries,spoken_languages,imdbId,year,genres_y,keywords
0,862.0,Toy Story,30000000,401157969,1995-11-22,81,en,21.5820,7.971,19651,"Family, Comedy, Animation, Adventure",United States of America,English,114709,1995.0,Adventure|Animation|Children|Comedy|Fantasy,"toys, computer animation, pixar animation, ani..."
1,8844.0,Jumanji,65000000,262821940,1995-12-15,104,en,3.1836,7.245,11172,"Adventure, Fantasy, Family",United States of America,"English, Français",113497,1995.0,Adventure|Children|Fantasy,"adventure, children, fantasy, kids, childhood"
2,15602.0,Grumpier Old Men,25000000,71518503,1995-12-22,101,en,2.1246,6.500,420,"Romance, Comedy",United States of America,English,113228,1995.0,Comedy|Romance,"good sequel, sequel, sequels, comedy, original"
3,31357.0,Waiting to Exhale,16000000,81452156,1995-12-22,127,en,2.0018,6.253,194,"Comedy, Drama, Romance",United States of America,English,114885,1995.0,Comedy|Drama|Romance,"women, chick flick, girlie movie, romantic, di..."
4,11862.0,Father of the Bride Part II,0,76594107,1995-12-08,106,en,2.5346,6.275,795,"Comedy, Family",United States of America,English,113041,1995.0,Comedy,"good sequel, sequel, sequels, midlife crisis, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57733,385377.0,Ballet of Blood,0,0,2016-03-01,98,en,0.3408,1.800,10,Horror,United States of America,English,3061100,2016.0,Horror,NaN
57734,644612.0,Letters Of Happiness,0,0,2019-12-03,103,ru,0.7063,6.300,9,"Drama, Family",Russia,Pусский,12073988,2019.0,Children|Drama,NaN
57735,845861.0,Akshardham: Operation Vajra Shakti,0,0,2025-07-04,106,hi,1.2825,10.000,2,"Action, Drama",India,हिन्दी,14418234,2021.0,Action|Drama,NaN
57736,878958.0,Ouija Japan,0,0,2021-10-19,78,en,0.6105,2.000,1,"Action, Horror",NaN,"日本語, English",11162178,2021.0,Action|Horror,NaN


In [5]:
genres1 = df['genres_x'].str.lower().str.replace(' ', '')
genres2 = df['genres_y'].str.lower().str.replace('|', ',')
df['genres'] = (
    (genres1 + ',' + genres2)
    .str.split(',')
    .explode()
    .dropna()
    .str.strip()
    .loc[lambda x: x != '']
    .groupby(level=0)
    .unique()
    .apply(lambda x: ','.join(x))
)
df.drop(['genres_x', 'genres_y', 'imdbId'], axis=1, inplace=True)
df['genres']

0        family,comedy,animation,adventure,children,fan...
1                        adventure,fantasy,family,children
2                                           romance,comedy
3                                     comedy,drama,romance
4                                            comedy,family
                               ...                        
57733                                               horror
57734                                drama,family,children
57735                                         action,drama
57736                                        action,horror
57737                                          documentary
Name: genres, Length: 57738, dtype: object

Połączenie z dodatkowo pobranymi filmami z bazy TMDB, których nie było w MovieLens

In [6]:
tmdb1 = pd.read_csv('../data/tmdb/tmdb_data.csv')
tmdb1['keywords'] = tmdb1['keywords'].str.replace(', ', ',')
tmdb1['genres'] = tmdb1['genres'].str.replace(', ', ',').str.lower()
tmdb1['release_date'] = pd.to_datetime(tmdb1['release_date'])
tmdb1['year'] = tmdb1['release_date'].dt.year
tmdb1

,Unnamed: 0,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,genres,origin_countries,spoken_languages,keywords,year
0,0,1284,Torrente 2: Mission in Marbella,4000000,21000000,2001-03-30,99,es,6.3480,6.000,196,"action,comedy,crime",Spain,Español,"missile,detective,roulette,marbella",2001.0
1,1,268590,"Vino Santo – Es lebe die Liebe, es lebe der Wein",0,0,2001-01-01,90,de,1.4360,5.000,1,NaN,NaN,Deutsch,NaN,2001.0
2,2,304139,Rahul,0,0,2001-03-23,151,hi,7.9703,3.500,2,drama,NaN,हिन्दी,NaN,2001.0
3,3,18269,Lady and the Tramp II: Scamp's Adventure,0,0,2001-02-18,69,en,2.9139,6.300,1305,"animation,family,romance,adventure","Australia, United States of America",English,"cartoon,sequel,initiation,cartoon dog,dogcatch...",2001.0
4,4,123747,Salad Days,0,0,2001-01-01,20,es,3.9037,8.000,1,comedy,Spain,Español,"short film,corto",2001.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376790,34904,1336841,Flieg steil,0,0,2024-09-12,86,de,2.2471,0.000,0,drama,Germany,Deutsch,NaN,2024.0
376791,35075,1477114,Sorority,0,0,2025-06-20,0,tl,7.7885,6.300,6,"drama,romance",Philippines,NaN,"softcore,lesbian,girls' love (gl),sexy",2025.0
376792,35193,1404535,Sherlock Holmes Mare of the Night,0,0,2025-01-24,0,en,3.9037,1.000,1,"horror,mystery,crime",NaN,English,NaN,2025.0
376793,35308,784524,Magazine Dreams,0,1076099,2025-03-21,123,en,2.2882,6.737,78,drama,United States of America,English,"bodybuilding,character study",2025.0


In [7]:
df = pd.concat([df, tmdb1], ignore_index=True)
print(len(df))
df.drop_duplicates(inplace=True)
df

434533


,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,origin_countries,spoken_languages,year,keywords,genres,Unnamed: 0
0,862.0,Toy Story,30000000,401157969,1995-11-22,81,en,21.5820,7.971,19651,United States of America,English,1995.0,"toys, computer animation, pixar animation, ani...","family,comedy,animation,adventure,children,fan...",NaN
1,8844.0,Jumanji,65000000,262821940,1995-12-15,104,en,3.1836,7.245,11172,United States of America,"English, Français",1995.0,"adventure, children, fantasy, kids, childhood","adventure,fantasy,family,children",NaN
2,15602.0,Grumpier Old Men,25000000,71518503,1995-12-22,101,en,2.1246,6.500,420,United States of America,English,1995.0,"good sequel, sequel, sequels, comedy, original","romance,comedy",NaN
3,31357.0,Waiting to Exhale,16000000,81452156,1995-12-22,127,en,2.0018,6.253,194,United States of America,English,1995.0,"women, chick flick, girlie movie, romantic, di...","comedy,drama,romance",NaN
4,11862.0,Father of the Bride Part II,0,76594107,1995-12-08,106,en,2.5346,6.275,795,United States of America,English,1995.0,"good sequel, sequel, sequels, midlife crisis, ...","comedy,family",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434528,1336841.0,Flieg steil,0,0,2024-09-12,86,de,2.2471,0.000,0,Germany,Deutsch,2024.0,NaN,drama,34904.0
434529,1477114.0,Sorority,0,0,2025-06-20,0,tl,7.7885,6.300,6,Philippines,NaN,2025.0,"softcore,lesbian,girls' love (gl),sexy","drama,romance",35075.0
434530,1404535.0,Sherlock Holmes Mare of the Night,0,0,2025-01-24,0,en,3.9037,1.000,1,NaN,English,2025.0,NaN,"horror,mystery,crime",35193.0
434531,784524.0,Magazine Dreams,0,1076099,2025-03-21,123,en,2.2882,6.737,78,United States of America,English,2025.0,"bodybuilding,character study",drama,35308.0


Połączenie danych o ludziach

In [8]:
people = pd.read_csv('../data/tmdb/tmdb_people_movieLens.csv')
print(len(people))
people1 = pd.read_csv('../data/tmdb/tmdb_people.csv')
print(len(people1))
people = pd.concat([people, people1], ignore_index=True)
people

408979
2060095


,movie_id,person_id,name,gender,known_for_department,popularity,job,Unnamed: 0
0,862.0,31,Tom Hanks,2,Acting,20.1613,actor,NaN
1,862.0,12898,Tim Allen,2,Acting,2.9738,actor,NaN
2,862.0,7167,Don Rickles,2,Acting,2.1703,actor,NaN
3,862.0,12899,Jim Varney,2,Acting,0.9327,actor,NaN
4,862.0,12900,Wallace Shawn,2,Acting,3.2563,actor,NaN
...,...,...,...,...,...,...,...,...
2469069,556574.0,1254614,Leslie Odom Jr.,2,Acting,1.2418,actor,264762.0
2469070,556574.0,87932,Renée Elise Goldsberry,1,Acting,1.8793,actor,264763.0
2469071,556574.0,1631814,Phillipa Soo,1,Acting,1.8363,actor,264764.0
2469072,556574.0,1652371,Daveed Diggs,2,Acting,2.1363,actor,264765.0


Do każdego filmu po jednym najpopularniejszym reżyserze, scenarzyście i producencie

In [9]:
single_choice_departments = ['Production', 'Writing', 'Directing']
actors = people[people['known_for_department'] == 'Acting']
others = people[people['known_for_department'].isin(single_choice_departments)]
others_top = others.loc[others.groupby(['movie_id', 'known_for_department'])['popularity'].idxmax()]
people = pd.concat([actors, others_top], ignore_index=True)
people['known_for_department'] = people['known_for_department'].replace({
    'Acting': 'actor',
    'Writing': 'writer',
    'Directing': 'director',
    'Production': 'producer',
})
people.drop(columns=['job'], inplace=True)
people.rename(columns={'known_for_department': 'job', 'person_id': 'id'}, inplace=True)
people

,movie_id,id,name,gender,job,popularity,Unnamed: 0
0,862.0,31,Tom Hanks,2,actor,20.1613,NaN
1,862.0,12898,Tim Allen,2,actor,2.9738,NaN
2,862.0,7167,Don Rickles,2,actor,2.1703,NaN
3,862.0,12899,Jim Varney,2,actor,0.9327,NaN
4,862.0,12900,Wallace Shawn,2,actor,3.2563,NaN
...,...,...,...,...,...,...,...
2244209,1656206.0,6089685,Teiichi Hori,0,writer,0.0000,283366.0
2244210,1656223.0,230455,Hiroyuki Kawasaki,2,director,1.3052,189292.0
2244211,1656223.0,230456,Ribon Kawasaki,0,writer,1.2770,189295.0
2244212,1656228.0,5583489,Karen Kopy,0,director,0.0314,146454.0


Do każdego filmu 5 najpopularniejszych aktorów - piwot danych (1 to najbardziej popularny)

In [10]:
actors = people[people['job'] == 'actor'].copy()
actors['rank'] = actors.groupby('movie_id')['popularity'].rank(method='first', ascending=False)
actors = actors[actors['rank'] <= 5]
actors['rank'] = actors['rank'].astype(int)

rows = []
for movie_id, group in actors.groupby("movie_id"):
    row = {"movie_id": movie_id}
    for i, person in group.iterrows():
        row[f"actor{person['rank']}_id"] = person['id']
        row[f"actor{person['rank']}_name"] = person['name']
        row[f"actor{person['rank']}_popularity"] = person['popularity']
        row[f"actor{person['rank']}_gender"] = person['gender']
    rows.append(row)
actors = pd.DataFrame(rows)
actors

,movie_id,actor4_id,actor4_name,actor4_popularity,actor4_gender,actor2_id,actor2_name,actor2_popularity,actor2_gender,actor1_id,...,actor1_popularity,actor1_gender,actor5_id,actor5_name,actor5_popularity,actor5_gender,actor3_id,actor3_name,actor3_popularity,actor3_gender
0,2.0,54768.0,Turo Pajala,0.7108,2.0,54769.0,Susanna Haavisto,1.1708,1.0,4826,...,1.8811,2,54770.0,Eetu Hilkamo,0.4084,2.0,1177496.0,Erkki Pajala,0.8157,2.0
1,3.0,4828.0,Sakari Kuosmanen,0.4103,2.0,5999.0,Kati Outinen,1.4703,1.0,4826,...,1.8811,2,1086499.0,Kylli Köngäs,0.1686,1.0,53508.0,Esko Nikkari,0.9185,2.0
2,5.0,3130.0,Jennifer Beals,3.7741,1.0,3129.0,Tim Roth,4.9629,2.0,3129,...,5.6589,2,3125.0,Madonna,2.6797,1.0,3130.0,Jennifer Beals,3.9675,1.0
3,6.0,9777.0,Cuba Gooding Jr.,2.6821,2.0,5724.0,Denis Leary,2.9368,2.0,5724,...,3.2793,2,9777.0,Cuba Gooding Jr.,2.5604,2.0,12799.0,Jeremy Piven,2.7546,2.0
4,9.0,NaN,NaN,NaN,NaN,902.0,Milton Welsh,0.5229,2.0,901,...,1.0757,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372456,1656202.0,6089677.0,Masato Tahara,0.0000,0.0,6086508.0,Makoto Sugimoto,0.0143,0.0,1118790,...,1.6712,1,NaN,NaN,NaN,NaN,5736226.0,Yu Sasaki,0.0071,0.0
372457,1656206.0,6089689.0,Maki Naitô,0.0000,0.0,1118805.0,Kyoko Hayami,0.4496,0.0,80131,...,1.1586,2,6089690.0,Kôji Nîzaki,0.0000,0.0,1202956.0,Ganko Fuyu,0.2431,0.0
372458,1656223.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3197490,...,0.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
372459,1656228.0,NaN,NaN,NaN,NaN,6089753.0,Tre Rodgers,0.0000,0.0,6089752,...,0.0000,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Piwot pozostałych osób

In [11]:
others = people[people['job'] != 'actor'].copy()
rows = []
for movie_id, group in others.groupby("movie_id"):
    row = {"movie_id": movie_id}
    for i, person in group.iterrows():
        row[f"{person['job']}_id"] = person['id']
        row[f"{person['job']}_name"] = person['name']
        row[f"{person['job']}_popularity"] = person['popularity']
        row[f"{person['job']}_gender"] = person['gender']
    rows.append(row)
others = pd.DataFrame(rows)
others

,movie_id,director_id,director_name,director_popularity,director_gender,writer_id,writer_name,writer_popularity,writer_gender,producer_id,producer_name,producer_popularity,producer_gender
0,2.0,16767.0,Aki Kaurismäki,1.9146,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.0,16767.0,Aki Kaurismäki,1.9146,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5.0,138.0,Quentin Tarantino,7.3808,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6.0,2042.0,Stephen Hopkins,1.5159,2.0,52035.0,Lewis Colick,2.3831,2.0,NaN,NaN,NaN,NaN
4,8.0,801.0,Michael Glawogger,1.6846,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
368735,1656187.0,1600946.0,Yukio Kitazawa,0.5924,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
368736,1656202.0,2051930.0,Masaharu Ito,0.7370,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
368737,1656206.0,1600946.0,Yukio Kitazawa,0.5924,2.0,6089685.0,Teiichi Hori,0.0000,0.0,NaN,NaN,NaN,NaN
368738,1656223.0,230455.0,Hiroyuki Kawasaki,1.3052,2.0,230456.0,Ribon Kawasaki,1.2770,0.0,NaN,NaN,NaN,NaN


Połączenie danych o ludziach i filmach

In [12]:
df = pd.merge(
    left=df,
    right=others,
    how='left',
    left_on='tmdbId',
    right_on='movie_id',
)
df = pd.merge(
    left=df,
    right=actors,
    how='left',
    left_on='tmdbId',
    right_on='movie_id',
)
df = df.drop(columns=['movie_id_x', 'movie_id_y'])
df.rename(columns={'top_5_tags': 'keywords'}, inplace=True)
df['keywords'] = df['keywords'].str.replace(', ', ',')
df

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,actor1_popularity,actor1_gender,actor5_id,actor5_name,actor5_popularity,actor5_gender,actor3_id,actor3_name,actor3_popularity,actor3_gender
0,862.0,Toy Story,30000000,401157969,1995-11-22,81,en,21.5820,7.971,19651,...,20.1613,2.0,12898.0,Tim Allen,2.9738,2.0,12900.0,Wallace Shawn,3.2563,2.0
1,8844.0,Jumanji,65000000,262821940,1995-12-15,104,en,3.1836,7.245,11172,...,7.7741,1.0,5149.0,Bonnie Hunt,2.6595,1.0,2157.0,Robin Williams,5.0475,2.0
2,15602.0,Grumpier Old Men,25000000,71518503,1995-12-22,101,en,2.1246,6.500,420,...,4.9967,1.0,589.0,Daryl Hannah,3.0548,1.0,589.0,Daryl Hannah,3.3151,1.0
3,31357.0,Waiting to Exhale,16000000,81452156,1995-12-22,127,en,2.0018,6.253,194,...,6.9280,2.0,8851.0,Whitney Houston,2.5071,1.0,9780.0,Angela Bassett,3.9122,1.0
4,11862.0,Father of the Bride Part II,0,76594107,1995-12-08,106,en,2.5346,6.275,795,...,3.9430,1.0,519.0,Martin Short,2.5075,2.0,67773.0,Steve Martin,3.5681,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434520,1336841.0,Flieg steil,0,0,2024-09-12,86,de,2.2471,0.000,0,...,1.7053,2.0,544813.0,Ceci Chuh,0.5405,1.0,1327557.0,Andreas Anke,0.7001,2.0
434521,1477114.0,Sorority,0,0,2025-06-20,0,tl,7.7885,6.300,6,...,5.1173,1.0,3378221.0,Sigrid Polon,0.6398,1.0,5078779.0,Julianne Richards,1.3810,1.0
434522,1404535.0,Sherlock Holmes Mare of the Night,0,0,2025-01-24,0,en,3.9037,1.000,1,...,1.1316,2.0,5127711.0,Dan Cortez,0.0610,0.0,3647725.0,Raven Heart,0.2102,1.0
434523,784524.0,Magazine Dreams,0,1076099,2025-03-21,123,en,2.2882,6.737,78,...,3.1156,2.0,58754.0,Haley Bennett,2.6153,1.0,58754.0,Haley Bennett,3.0705,1.0


Uwzględnienie inflacji - dane są w dolarach amerykańskich, a więc uwzględniamy inflację tej waluty. Jako rok bazowy przyjmujemy 2025 rok i pod ten rok dostosowujemy wszystkie wartości.

In [13]:
cpi = pd.read_excel('../data/cpi.xlsx', skiprows=11)
cpi = cpi[['Year', 'Annual']].set_index('Year').to_dict()['Annual']
cpi

c:\Users\olkat\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


{1950: 24.1,
 1951: 26.0,
 1952: 26.5,
 1953: 26.7,
 1954: 26.9,
 1955: 26.8,
 1956: 27.2,
 1957: 28.1,
 1958: 28.9,
 1959: 29.1,
 1960: 29.6,
 1961: 29.9,
 1962: 30.2,
 1963: 30.6,
 1964: 31.0,
 1965: 31.5,
 1966: 32.4,
 1967: 33.4,
 1968: 34.8,
 1969: 36.7,
 1970: 38.8,
 1971: 40.5,
 1972: 41.8,
 1973: 44.4,
 1974: 49.3,
 1975: 53.8,
 1976: 56.9,
 1977: 60.6,
 1978: 65.2,
 1979: 72.6,
 1980: 82.4,
 1981: 90.9,
 1982: 96.5,
 1983: 99.6,
 1984: 103.9,
 1985: 107.6,
 1986: 109.6,
 1987: 113.6,
 1988: 118.3,
 1989: 124.0,
 1990: 130.7,
 1991: 136.2,
 1992: 140.3,
 1993: 144.5,
 1994: 148.2,
 1995: 152.4,
 1996: 156.9,
 1997: 160.5,
 1998: 163.0,
 1999: 166.6,
 2000: 172.2,
 2001: 177.1,
 2002: 179.9,
 2003: 184.0,
 2004: 188.9,
 2005: 195.3,
 2006: 201.6,
 2007: 207.342,
 2008: 215.303,
 2009: 214.537,
 2010: 218.056,
 2011: 224.939,
 2012: 229.594,
 2013: 232.957,
 2014: 236.736,
 2015: 237.017,
 2016: 240.007,
 2017: 245.12,
 2018: 251.107,
 2019: 255.657,
 2020: 258.811,
 2021: 270.97

In [14]:
base_year = 2025
data = df.copy()
data['budget_adjusted'] = round(df['budget'] * (cpi[base_year] / data['year'].map(cpi)), 2)
data['revenue_adjusted'] = round(df['revenue'] * (cpi[base_year] / data['year'].map(cpi)), 2)
data

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,actor5_id,actor5_name,actor5_popularity,actor5_gender,actor3_id,actor3_name,actor3_popularity,actor3_gender,budget_adjusted,revenue_adjusted
0,862.0,Toy Story,30000000,401157969,1995-11-22,81,en,21.5820,7.971,19651,...,12898.0,Tim Allen,2.9738,2.0,12900.0,Wallace Shawn,3.2563,2.0,6.337461e+07,8.474409e+08
1,8844.0,Jumanji,65000000,262821940,1995-12-15,104,en,3.1836,7.245,11172,...,5149.0,Bonnie Hunt,2.6595,1.0,2157.0,Robin Williams,5.0475,2.0,1.373116e+08,5.552079e+08
2,15602.0,Grumpier Old Men,25000000,71518503,1995-12-22,101,en,2.1246,6.500,420,...,589.0,Daryl Hannah,3.0548,1.0,589.0,Daryl Hannah,3.3151,1.0,5.281217e+07,1.510819e+08
3,31357.0,Waiting to Exhale,16000000,81452156,1995-12-22,127,en,2.0018,6.253,194,...,8851.0,Whitney Houston,2.5071,1.0,9780.0,Angela Bassett,3.9122,1.0,3.379979e+07,1.720666e+08
4,11862.0,Father of the Bride Part II,0,76594107,1995-12-08,106,en,2.5346,6.275,795,...,519.0,Martin Short,2.5075,2.0,67773.0,Steve Martin,3.5681,2.0,0.000000e+00,1.618040e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434520,1336841.0,Flieg steil,0,0,2024-09-12,86,de,2.2471,0.000,0,...,544813.0,Ceci Chuh,0.5405,1.0,1327557.0,Andreas Anke,0.7001,2.0,0.000000e+00,0.000000e+00
434521,1477114.0,Sorority,0,0,2025-06-20,0,tl,7.7885,6.300,6,...,3378221.0,Sigrid Polon,0.6398,1.0,5078779.0,Julianne Richards,1.3810,1.0,0.000000e+00,0.000000e+00
434522,1404535.0,Sherlock Holmes Mare of the Night,0,0,2025-01-24,0,en,3.9037,1.000,1,...,5127711.0,Dan Cortez,0.0610,0.0,3647725.0,Raven Heart,0.2102,1.0,0.000000e+00,0.000000e+00
434523,784524.0,Magazine Dreams,0,1076099,2025-03-21,123,en,2.2882,6.737,78,...,58754.0,Haley Bennett,2.6153,1.0,58754.0,Haley Bennett,3.0705,1.0,0.000000e+00,1.076099e+06


Usuwanie duplikatów tych samych tytułów, tmdbId i roku produkcji

In [15]:
data['keywords_count'] = data['keywords'].fillna('').str.count(',')
data = data.sort_values(by='keywords_count', ascending=True)
data = data.drop(columns=['keywords_count'])

data = data.drop_duplicates(subset=['tmdbId', 'title', 'release_date'], keep='first')
data

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,actor5_id,actor5_name,actor5_popularity,actor5_gender,actor3_id,actor3_name,actor3_popularity,actor3_gender,budget_adjusted,revenue_adjusted
321260,531700.0,There Goes Our Hero,0,0,1979-11-03,95,ja,1.2601,6.000,2,...,551578.0,Terumi Niki,0.9051,1.0,1238594.0,Kaneta Kimotsuki,1.2973,2.0,0.00,0.000000e+00
321257,1378008.0,Madame Sourdis,0,0,1979-12-31,0,fr,1.2558,0.000,0,...,1166156.0,Marcel Imhoff,0.9346,2.0,20118.0,Françoise Brion,1.0139,1.0,0.00,0.000000e+00
321252,327702.0,Tesoro mio,0,0,1979-12-22,104,it,1.0517,3.857,7,...,32377.0,Enrico Maria Salerno,0.6027,2.0,119991.0,Renato Pozzetto,1.2232,2.0,0.00,0.000000e+00
321248,380548.0,Mamang Sorbetero,0,0,1979-12-25,0,tl,1.0241,0.000,0,...,1360065.0,Rod Navarro,0.3930,0.0,1547679.0,Celeste Legaspi,0.9112,0.0,0.00,0.000000e+00
321244,75268.0,Stories Our Nannies Don't Tell,0,0,1979-11-17,97,pt,1.2597,5.150,10,...,525716.0,Meiry Vieira,0.3484,1.0,287678.0,Adele Fátima,0.6302,1.0,0.00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230300,1324139.0,Namas Dei: The Tucker J. James Story,0,0,2025-09-25,100,en,0.4782,0.000,0,...,4862145.0,Laura Juliane May,0.0586,1.0,2027042.0,Carrie Gibson,0.4994,1.0,0.00,0.000000e+00
227331,1467523.0,Rooftop Lempicka,0,0,2025-03-15,20,vi,0.5036,0.000,0,...,5383064.0,Ngô Phương Thảo,0.0143,0.0,1599725.0,Hoàng Trần Minh Đức,0.0256,0.0,0.00,0.000000e+00
222917,1315711.0,Clocking The T,88000,0,2024-07-11,105,en,0.6564,0.000,0,...,1278781.0,Rebecca Reaney,0.3502,1.0,3373709.0,Ben Hicks,0.5745,0.0,90315.52,0.000000e+00
219855,918096.0,"Hemet, or the Landlady Don't Drink Tea",29000,0,2024-02-24,89,en,0.6337,4.250,4,...,NaN,NaN,NaN,NaN,3353848.0,Kimberly Weinberger,0.6222,1.0,29763.07,0.000000e+00


In [16]:
data = data.drop('Unnamed: 0', axis = 1)

In [17]:
# Będziemy wykorzystywać tylko te, gdzie revenue > 0, bo jest to nasza zmienna wynikowa
data = data[(data['revenue_adjusted'] > 0)]
data.shape

(20511, 49)

In [18]:
data.isna().sum()

tmdbId                     0
title                      0
budget                     0
revenue                    0
release_date               0
runtime                    0
original_language          0
popularity                 0
vote_average               0
vote_count                 0
origin_countries         370
spoken_languages         341
year                       0
keywords                6562
genres                    89
director_id             4391
director_name           4391
director_popularity     4391
director_gender         4391
writer_id               6531
writer_name             6531
writer_popularity       6531
writer_gender           6531
producer_id            19098
producer_name          19098
producer_popularity    19098
producer_gender        19098
actor4_id                645
actor4_name              645
actor4_popularity        645
actor4_gender            645
actor2_id                256
actor2_name              256
actor2_popularity        256
actor2_gender 

In [19]:
# Usunięcie producenta, bo za dużo braków w danych
data = data.drop(['producer_id', 'producer_name', 'producer_popularity', 'producer_gender'], axis = 1)

Połączenie danych ostatni raz - z ramką tmdb_missing.csv, która próbowała zebrać informacje z TMDB tylko o brakujących danych (keywords, reżyserze i scenarzyście, bo ich było najwięcej)

In [20]:
missing = pd.read_csv("../data/tmdb/tmdb_missing.csv")

In [21]:
data_copy = data.copy()

In [22]:
missing.set_index('tmdbId', inplace=True)
data_copy.set_index('tmdbId', inplace=True)
data_copy = data_copy.combine_first(missing)
data_copy.reset_index(inplace=True)

In [23]:
data_copy = data_copy[data.columns]

In [24]:
data_copy.isna().sum()  # po połączeniu udało się wiele braków danych uzupełnić

tmdbId                    0
title                     0
budget                    0
revenue                   0
release_date              0
runtime                   0
original_language         0
popularity                0
vote_average              0
vote_count                0
origin_countries        370
spoken_languages        341
year                      0
keywords               3179
genres                   89
director_id              82
director_name            82
director_popularity      82
director_gender          82
writer_id               923
writer_name             923
writer_popularity       923
writer_gender           923
actor4_id               645
actor4_name             645
actor4_popularity       645
actor4_gender           645
actor2_id               256
actor2_name             256
actor2_popularity       256
actor2_gender           256
actor1_id               129
actor1_name             129
actor1_popularity       129
actor1_gender           129
actor5_id           

In [25]:
data_copy.to_csv('../data/merged/merged_data.csv', index = False)